# Chapter 4: The Development Method

Code of Chapter 4 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

## 4.1.2 The example triangle

In [ ]:
import numpy as np
import pandas as pd
from gidevelop import age_to_age, average_factors, cumulative_factors, develop

T = pd.DataFrame([[1000, 1500, 1650, 1700], [1100, 1700, 1850, np.nan],
                  [1200, 1750, np.nan, np.nan], [1300, np.nan, np.nan, np.nan]],
                 index=pd.Index([2022, 2023, 2024, 2025], name="accident_year"),
                 columns=pd.Index([12, 24, 36, 48], name="age"))
age_to_age(T).round(4)

## 4.2.1 Averages

In [ ]:
for m in ["simple", "volume", "medial", "geometric"]:
    print(m, average_factors(T, m).round(4).tolist())
# simple [1.5013, 1.0941, 1.0303], volume [1.5, 1.0938, 1.0303], ...

## 4.3.4 Worked example

In [ ]:
sel = average_factors(T, "volume")                 # 1.5000, 1.0938, 1.0303
cdf = cumulative_factors(sel, tail=1.0)            # 1.6903, 1.1269, 1.0303, 1.0000
res = develop(T, cdf)
res["ultimate"].sum()                              # 7775.57

In [ ]:
P_ex = pd.DataFrame([[500, 1100, 1450, 1650], [550, 1150, 1600, np.nan],
                     [560, 1200, np.nan, np.nan], [600, np.nan, np.nan, np.nan]],
                    index=T.index, columns=T.columns)
paid_cdf = cumulative_factors(average_factors(P_ex, "volume"), tail=1700 / 1650)
develop(P_ex, paid_cdf)["ultimate"].sum()          # 7526.34

## 4.4.2 Results, and a comparison with the truth

In [ ]:
from gitriangle import synthetic_history, triangle
pol, cl, txs = synthetic_history(2020, 2025, seed=100)
I = triangle(cl, txs, "incurred", 2021, 2025)
P = triangle(cl, txs, "paid", 2021, 2025)
ult_I = develop(I, cumulative_factors(average_factors(I, "volume"), 1.0))["ultimate"]
ult_P = develop(P, cumulative_factors(average_factors(P, "volume"), 1.0))["ultimate"]
ult_I.sum(), ult_P.sum()                           # 12953070, 12765955

In [ ]:
from gidevelop import fit_exponential_tail
fit_exponential_tail(average_factors(P, "volume"))   # a=3.657, b=1.601, tail=1.0015

## 4.4.3 The reserve

In [ ]:
from gidevelop import reserve_summary
rs = reserve_summary(P, I, ult_I)
rs[["case_reserve", "ibnr", "unpaid"]].sum()       # 1947963, 402044, 2350007

## 4.4.4 Checking against chainladder

In [ ]:
try:
    import chainladder  # optional: not available in the browser version
except ImportError:
    chainladder = None
    print("chainladder is not installed: this cross-check is skipped.")
if chainladder is not None:
    import chainladder as chl
    from gitriangle import to_long
    tri = chl.Triangle(to_long(I, "incurred"), origin="accident_year",
                       development="valuation", columns=["incurred"], cumulative=True)
    model = chl.Chainladder().fit(chl.Development(average="volume").fit_transform(tri))
    model.ultimate_.to_frame().round(0).values.ravel().tolist()
    # [2325132.0, 2131796.0, 2351184.0, 2935653.0, 3209305.0]

## Exercise 1: A projection from a new triangle

In [ ]:
X = pd.DataFrame([[800, 1040, 1144, 1167, 1170], [900, 1197, 1305, 1331, np.nan],
                  [950, 1235, 1359, np.nan, np.nan], [1000, 1310, np.nan, np.nan, np.nan],
                  [1100, np.nan, np.nan, np.nan, np.nan]],
                 index=pd.Index(range(2021, 2026), name="accident_year"),
                 columns=pd.Index([12, 24, 36, 48, 60], name="age"))
develop(X, cumulative_factors(average_factors(X, "volume"), 1.0))["ultimate"].sum()   # 6979.85

## Exercise 2: The leverage of the tail factor

In [ ]:
res2 = develop(T, cumulative_factors(average_factors(T, "volume"), tail=1.02))
res2["ultimate"].sum()                             # 7931.08

## Exercise 3: Latest-three-years factors on the synthetic portfolio

In [ ]:
cdf3 = cumulative_factors(average_factors(I, "volume", n=3), 1.0)
develop(I, cdf3).loc[2025, "ultimate"]             # 3240470

## 4.8 Python Walkthrough

In [ ]:
from gidevelop import (age_to_age, average_factors, cumulative_factors,
                       latest_diagonal, develop, reserve_summary)

# Sections 4.1-4.3: the example triangle T
age_to_age(T).round(4)                             # 1.5000, 1.5455, 1.4583, ...
cdf = cumulative_factors(average_factors(T, "volume"), tail=1.0)
develop(T, cdf)[["cdf", "pct_developed", "ultimate"]].round(4)

# Section 4.4: the synthetic portfolio (I, P from synthetic_history)
ult_I = develop(I, cumulative_factors(average_factors(I, "volume"), 1.0))["ultimate"]
rs = reserve_summary(P, I, ult_I)
rs["unpaid"].sum().round(0)                        # 2350007.0